# Accuracy Says 99%, and the Model Is Useless

**Topic:** Imbalanced classification &nbsp;·&nbsp; **Difficulty:** Medium

Fraud is 1.2% of transactions. A model that predicts "never fraud" scores 98.8%
accuracy and catches nothing. The default 0.5 threshold on a logistic model
scores almost the same accuracy and still misses most frauds, because with this
base rate almost no predicted probability ever exceeds 0.5.

Keep the model, move the threshold. The probabilities were fine. The decision
rule was not.

## The data (created in the next cell)

- `X_train` `(6000, 12)`, `y_train`: about 1.2% positives.
- `X_test` `(4000, 12)`, `y_test`: same rate.

## Your task

Using **sklearn's `LogisticRegression(max_iter=2000)`** and NumPy:

1. Fit on the training set and get predicted probabilities for the test set.
2. `acc_default` and `f1_default`: accuracy and F1 at threshold 0.5.
3. Sweep thresholds `np.linspace(0.01, 0.99, 99)` and find the one maximizing
   **test F1** and call it `best_t`. Report `f1_best` and `acc_best` there.
4. Set `answers` to a dict with `"acc_default"`, `"f1_default"`, `"best_t"`,
   `"f1_best"`, `"acc_best"`.
   Define F1 as `0` when the model predicts no positives at all.

## Evaluation

Compared to the reference to a relative tolerance of 1e-6. The evaluation cell
checks the tuned F1 beats the default and that accuracy did not go *up*.

In [ ]:
import numpy as np

rng = np.random.default_rng(606)

def make(n, seed):
    r = np.random.default_rng(seed)
    y = (r.random(n) < 0.012).astype(int)
    X = r.normal(0, 1, (n, 12))
    X[:, :4] += y[:, None] * 1.6
    return X, y

X_train, y_train = make(6000, 21)
X_test, y_test = make(4000, 22)
print(f"train positives: {y_train.mean():.4f}, test positives: {y_test.mean():.4f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("acc_default", [], 0.991, -0.16593107268116866, 1e-06),
    ("f1_default", [], 0.45454545454545453, -0.07610818855204507, 1e-06),
    ("best_t", [], 0.13, -0.02176694192588489, 1e-06),
    ("f1_best", [], 0.6226415094339622, -0.10425385828072589, 1e-06),
    ("acc_best", [], 0.99, -0.16576363466635416, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "f1_best" in answers and "f1_default" in answers:
    _up = answers["f1_best"] > answers["f1_default"]
    _down = answers["acc_best"] <= answers["acc_default"]
    print(f"  {'F1 up, accuracy down':<24} {'ok' if _up and _down else 'wrong'}  "
          f"(F1 {answers['f1_default']:.3f}->{answers['f1_best']:.3f}, "
          f"acc {answers['acc_default']:.4f}->{answers['acc_best']:.4f})")
    _ok = _ok and _up and _down

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Accuracy going *down* as F1 goes up is the point: you are trading a large number
of cheap false positives for a small number of expensive false negatives, which
is the right trade when a missed fraud costs a hundred times a manual review.
The threshold is a business parameter, not a model parameter, and 0.5 is only
correct when the classes are balanced and the errors cost the same.

```python
import numpy as np
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=2000).fit(X_train, y_train)
prob = model.predict_proba(X_test)[:, 1]

def f1_of(pred):
    tp = float(((pred == 1) & (y_test == 1)).sum())
    fp = float(((pred == 1) & (y_test == 0)).sum())
    fn = float(((pred == 0) & (y_test == 1)).sum())
    if tp == 0:
        return 0.0
    prec, rec = tp / (tp + fp), tp / (tp + fn)
    return 2 * prec * rec / (prec + rec)

pred_d = (prob >= 0.5).astype(int)
acc_default = float((pred_d == y_test).mean())
f1_default = f1_of(pred_d)

ts = np.linspace(0.01, 0.99, 99)
f1s = np.array([f1_of((prob >= t).astype(int)) for t in ts])
j = int(np.argmax(f1s))
best_t = float(ts[j])
pred_b = (prob >= best_t).astype(int)

answers = {
    "acc_default": acc_default,
    "f1_default": f1_default,
    "best_t": best_t,
    "f1_best": float(f1s[j]),
    "acc_best": float((pred_b == y_test).mean()),
}
```